# Generate a voltage-time + spectrogram from a given velocity time history

### Input velocity vs time curve

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

from pdv_synthesis import (
    generate_velocity_profile,
    velocity_to_pdv_voltage,
    velocity_to_multipoint_pdv_voltage,
    compute_stft,
    load_waveform_csv,
)

OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Digitizer and noise settings
sample_rate = 128e9  # Hz
noise_sd = 0.1

# Choose shot type: "velocity", "HEL", "HEL-spall"
shot_type = "HEL"

time, vel_sim = generate_velocity_profile(shot_type, sample_rate=sample_rate)

# Plot result
plt.figure(figsize=(7, 3))
plt.plot(time * 1e9, vel_sim, lw=2)
plt.xlim([-10, 100])
plt.xlabel("Time (ns)")
plt.ylabel("Velocity (m/s)")
plt.title(f"Synthetic Free-Surface Velocity ({shot_type})")
plt.tight_layout()
plt.show()

### Convert into position to get phase difference
- This is wrt the target laser (would be the phase difference for a homodyne system)

$$
\phi(t) = \frac{4 \pi \Delta L(t) \,}{\lambda}
$$

- Calculate the difference in phi by setting L_0 to equal 0. The position therefore equals L

In [ ]:
# Position and phase are computed internally by velocity_to_pdv_voltage.
# Shown here for illustration only.
pos_sim = np.cumsum(vel_sim / sample_rate)

fig, ax = plt.subplots(1, 2)
ax[0].plot(time * 1e9, pos_sim * 1e6)
ax[0].set_xlabel("time (ns)")
ax[0].set_ylabel("position (microns)")
ax[0].set_xlim([-10, 100])

lambda_ref = 1550e-9
phase = 4 * np.pi * pos_sim / lambda_ref
phase_wrapped = np.remainder(phase, 2 * np.pi)
ax[1].plot(time * 1e9, phase_wrapped / np.pi)
ax[1].set_ylabel("phase (multiples of pi)")
ax[1].set_xlabel("time (ns)")
ax[1].set_xlim([-10, 100])

fig.tight_layout()

### Optical Field Definitions in Heterodyne PDV (single point)


#### **Laser Source Field**

$$
E_{tar} = E_{tar0} \cos(2\pi f_L t)
$$

This represents the **original laser field** emitted at optical frequency
$$
f_L = \frac{c}{\lambda}
$$
It is the laser field of the target before it is reflected off a moving surface. 

#### **Local Oscillator / Reference Field**

$$
E_{ref} = E_{ref0} \cos[2\pi (f_L + f_0)t]
$$

This is the **reference (local oscillator)** beam.  
It is derived from $ E_r $ and **frequency-shifted by $ f_0 $** (e.g., 2 GHz).  
It provides a **stable, known phase** for heterodyne mixing at the detector.


#### **Reflected (Signal) Field from the Moving Surface**

$$
E_s = E_{s0} \cos\!\left[2\pi f_L t - \frac{4\pi L(t)}{\lambda}\right]
$$

This is **signal** or **reflected beam** returning from the moving target surface.  
The term $ \frac{4\pi L(t)}{\lambda} $ represents the **round-trip phase shift** due to motion;  
its time derivative encodes the **Doppler velocity** of the surface.

#### **Combined Field at the Detector**

$$
E_{\text{total}} = E_{ref} + E_s
$$

The detector measures **intensity**, proportional to $ |E_{\text{total}}|^2 $,  
producing a **beat signal** at frequency $ f_0 \pm f_D $,  
where $ f_D = \frac{2v}{\lambda} $ is the Doppler shift from target motion.

---

#### **Detected Intensity (Beat Signal)**

$$
I(t) \propto E_{ref0}^2 + E_{s0}^2 + 2E_{ref0}E_{s0} 
\cos\!\left[2\pi f_0 t - \frac{4\pi L(t)}{\lambda}\right]
$$

The **cross term** contains the **heterodyne beat**.  
Its phase $ 2\pi f_0 t - \frac{4\pi L(t)}{\lambda} $ evolves with target displacement.  
The squared terms are DC signals automatically filtered out by oscilloscope

$$
I(t) \propto 2E_{ref0}E_{s0} 
\cos\!\left[2\pi f_0 t - \frac{4\pi L(t)}{\lambda}\right]
$$

---

#### **Constant Reflection Case**

$$
E_{\text{total}} = E_{ref} + E_s + E_{tar}
$$

Where $E_{tar}$ is the electromagnetic field of the original outgoing source field. This is the result of a backreflection off a stationary target. 

The same logic may be applied here: 

$$
I(t) \propto 2E_{ref0}E_{s0} + 2E_{ref0}E_{tar0} + 2E_{tar0}E_{s0}
$$




In [ ]:
single_point_csv = os.path.join(OUTPUT_DIR, "waveform.csv")

voltage = velocity_to_pdv_voltage(
    time,
    vel_sim,
    lambda_ref=1550e-9,
    lambda_tar=1550.016e-9,
    power_ref_dbm=0,
    power_signal_dbm=-1,
    noise_sd=noise_sd,
    output_csv=single_point_csv,
)

plt.plot(time * 1e9, voltage)
plt.xlabel("time (ns)")
plt.ylabel("voltage")
plt.xlim([-50, 100])
plt.title("Synthetic PDV Voltage (single-point)")
plt.tight_layout()
plt.show()

print(f"Saved → {single_point_csv}  ({len(voltage)} samples)")

### Generate Spectrogram

In [ ]:
nperseg = 600
noverlap = 500
nfft = 5120

f, t, Zxx = compute_stft(voltage, fs=sample_rate, window="hann",
                          nperseg=nperseg, noverlap=noverlap, nfft=nfft)

mag = np.abs(Zxx)
t_begin, t_end = time[0], time[-1]

plt.imshow(
    10 * np.log10(mag ** 2),
    aspect="auto",
    origin="lower",
    interpolation="none",
    extent=[t_begin / 1e-9, t_end / 1e-9, f[0] / 1e9, f[-1] / 1e9],
    cmap="viridis",
)
plt.colorbar(label="Power (dBm)")
plt.ylim([0, 5])
plt.xlim([-50, 200])
plt.xlabel("Time (ns)")
plt.ylabel("Frequency (GHz)")
plt.minorticks_on()
plt.title("Spectrogram — Single-Point PDV")

# Multipoint
- Same idea as before, but mixing for multiple frequencies
- There is no beat frequencies (or crosstalk) created between probes due to large spacing (eg 1531nm to 1537nm)

In [ ]:
multi_point_csv = os.path.join(OUTPUT_DIR, "waveform_multi.csv")

voltage_multi = velocity_to_multipoint_pdv_voltage(
    time,
    vel_sim,
    num_points=2,
    ref_lambdas_nm=[1531.116, 1537.397, 1543.730],
    tar_lambdas_nm=[1531.102, 1537.374, 1543.703],  # ~3, 7, 10 GHz below ref
    power_ref_dbm=8,
    power_signal_dbm=-10,
    noise_sd=noise_sd,
    output_csv=multi_point_csv,
)

plt.plot(time * 1e9, voltage_multi)
plt.xlabel("time (ns)")
plt.ylabel("voltage")
plt.xlim([0, 20])
plt.title("Synthetic PDV Voltage (multi-point)")
plt.tight_layout()
plt.show()

print(f"Saved → {multi_point_csv}  ({len(voltage_multi)} samples)")

In [ ]:
f, t, Zxx = compute_stft(voltage_multi, fs=sample_rate, window="hann",
                          nperseg=nperseg, noverlap=noverlap, nfft=nfft)

mag = np.abs(Zxx)

plt.imshow(
    10 * np.log10(mag ** 2),
    aspect="auto",
    origin="lower",
    interpolation="none",
    extent=[t_begin / 1e-9, t_end / 1e-9, f[0] / 1e9, f[-1] / 1e9],
    cmap="viridis",
)
plt.colorbar(label="Power (dBm)")
plt.ylim([0, 20])
plt.xlabel("Time (ns)")
plt.ylabel("Frequency (GHz)")
plt.minorticks_on()
plt.title("Spectrogram — Multi-Point PDV")